In [140]:
import os
import sys
import gtfparse
from matplotlib.backend_bases import NonGuiException
import numpy as np
import pandas as pd
from pathlib import Path
import logging
import json
import muon as mu

import torch
import argparse

PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/TETHER")
sys.path.append(str(PROJECT_DIR))

DATA_DIR = PROJECT_DIR.parent / "data"

import utils
from utils import prepare_tftg_lookup_tables, build_tftg_inputs

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def _format_chroms(chroms: list[str]) -> str:
    """Render a chromosome list for logging.

    Only for log messages -- the splits themselves are made with isin() on the full list.
    min()/max() would compare these as strings, where "9" > "15", so ["1".."15"] printed
    as "1-15" came out as "1-9". Numeric labels are ordered numerically and collapsed to
    a range only when they are actually contiguous.
    """
    numeric = sorted((c for c in chroms if str(c).isdigit()), key=int)
    other = sorted(str(c) for c in chroms if not str(c).isdigit())

    parts = []
    if numeric:
        contiguous = int(numeric[-1]) - int(numeric[0]) == len(numeric) - 1
        parts.append(f"{numeric[0]}-{numeric[-1]}" if contiguous and len(numeric) > 1
                     else ", ".join(numeric))
    parts.extend(other)
    return ", ".join(parts)


def split_genes_by_chromosome(
    gene_reference_file: Path,
    train_chroms: list[str] = None,
    val_chroms: list[str] = None,
    test_chroms: list[str] = None
    ):
    logging.info(f"Splitting genes into train/val/test based on chromosome:")
    gene_ref_df = gtfparse.read_gtf(gene_reference_file, result_type="pandas")

    gene_chrom: pd.DataFrame = gene_ref_df[["seqname", "gene_name"]].rename(
        columns={"seqname": "chrom", "gene_name": "TG"}
    )
    
    gene_chrom["chrom"] = gene_chrom["chrom"].astype(str).str.replace("^chr", "", regex=True)
    gene_chrom["TG"] = gene_chrom["TG"].str.upper()
    
    train_genes = gene_chrom[gene_chrom["chrom"].isin(train_chroms)][
        "TG"
    ].unique()
    logging.info(f"  - Train set: {len(train_genes):,} genes (chroms {_format_chroms(train_chroms)})")

    val_genes = gene_chrom[gene_chrom["chrom"].isin(val_chroms)][
        "TG"
    ].unique()
    logging.info(f"  - Validation set: {len(val_genes):,} genes (chroms {_format_chroms(val_chroms)})")

    test_genes = gene_chrom[gene_chrom["chrom"].isin(test_chroms)]["TG"].unique()
    logging.info(f"  - Test set: {len(test_genes):,} genes (chroms {_format_chroms(test_chroms)})")

    return train_genes, val_genes, test_genes


def create_train_val_test_splits(
    tf_tg_labeled_df: pd.DataFrame,
    train_genes: np.ndarray,
    val_genes: np.ndarray,
    test_genes: np.ndarray,
):
    train_genes_set = set(train_genes)
    val_genes_set = set(val_genes)
    test_genes_set = set(test_genes)

    # Subset the ground truth to create train/val/test splits based on the target gene chromosome splits
    gt_train_df = tf_tg_labeled_df[tf_tg_labeled_df["tg_id"].isin(train_genes_set)].copy()
    gt_val_df = tf_tg_labeled_df[tf_tg_labeled_df["tg_id"].isin(val_genes_set)].copy()
    gt_test_df = tf_tg_labeled_df[tf_tg_labeled_df["tg_id"].isin(test_genes_set)].copy()
    
    if len(gt_train_df) == 0:
        logging.warning("No training interactions found for the selected train genes.")
        logging.info(f"Dataset genes: {list(train_genes_set)[:5]}")
        logging.info(f"Ground truth target genes: {tf_tg_labeled_df['tg_id'].unique()[:5]}")

    return gt_train_df, gt_val_df, gt_test_df


def create_true_false_edges_from_full_universe(
    edge_df: pd.DataFrame,
    tf_col: str = "Source",
    item_col: str = "Target",
    pct_true_edges: float | None = 1.0,
    true_false_ratio: float = 1.0,
    seed: int = 123,
):
    df_all = edge_df[[tf_col, item_col]].copy()

    df_all = df_all.dropna(subset=[tf_col, item_col])

    df_all[tf_col] = df_all[tf_col].astype(str)
    df_all[item_col] = df_all[item_col].astype(str)

    df_all = df_all.drop_duplicates([tf_col, item_col]).reset_index(drop=True)

    if df_all.empty:
        raise ValueError(
            f"No edges remain after filtering by tf_names using columns "
            f"{tf_col!r} and {item_col!r}."
        )

    candidate_tfs = sorted(df_all[tf_col].unique())
    candidate_items = sorted(df_all[item_col].unique())

    gt_pairs = set(zip(df_all[tf_col], df_all[item_col]))

    full_universe = (
        pd.MultiIndex
        .from_product([candidate_tfs, candidate_items], names=[tf_col, item_col])
        .to_frame(index=False)
    )

    full_universe["_pair"] = list(zip(full_universe[tf_col], full_universe[item_col]))
    full_universe["_in_gt"] = full_universe["_pair"].isin(gt_pairs)

    true_df = full_universe[full_universe["_in_gt"]].copy()
    false_df = full_universe[~full_universe["_in_gt"]].copy()

    if pct_true_edges is not None:
        if not (0 < pct_true_edges <= 1):
            raise ValueError("pct_true_edges must be in (0, 1] or None.")

        true_df = true_df.sample(frac=pct_true_edges, random_state=seed)

    n_false = round(len(true_df) * true_false_ratio)

    if n_false > len(false_df):
        logging.warning(
            f"Requested {n_false:,} false edges, but only {len(false_df):,} are available. "
            "Using all available false edges."
        )
        n_false = len(false_df)

    false_df = false_df.sample(n=n_false, random_state=seed)

    true_edges = set(zip(true_df[tf_col], true_df[item_col]))
    false_edges = set(zip(false_df[tf_col], false_df[item_col]))

    return true_edges, false_edges


def create_labeled_tf_tg_dataset(
    true_interactions: set[tuple[str, str]],
    false_interactions: set[tuple[str, str]],
    tf_name_to_idx: dict[str, int],
    drop_missing: bool = True,
) -> pd.DataFrame:
    # sorted(), not bare set iteration: set order over string tuples depends on
    # PYTHONHASHSEED, so without this the col order -- and therefore anything
    # downstream that indexes by position, e.g. df.sample(n=...) -- differs in every
    # process even with a fixed random_state.
    rows = []
    for tf, tg in sorted(true_interactions):
        rows.append((tf, tg, 1))
    for tf, tg in sorted(false_interactions):
        rows.append((tf, tg, 0))

    df = pd.DataFrame(rows, columns=["tf_name", "tg_id", "label"])
    df["tf_embedding_idx"] = df["tf_name"].map(tf_name_to_idx)

    missing_mask = df["tf_embedding_idx"].isna()
    if missing_mask.any():
        n_missing = missing_mask.sum()
        if drop_missing:
            logging.info(f"Dropping {n_missing} interactions with missing TF or TG indices.")
            df = df.loc[~missing_mask].copy()
        else:
            missing_examples = df.loc[missing_mask].head()
            raise ValueError(
                f"{n_missing} interactions are missing TF or TG indices.\n"
                f"Examples:\n{missing_examples}"
            )

    df["tf_embedding_idx"] = df["tf_embedding_idx"].astype(np.int64)
    df["label"] = df["label"].astype(np.bool)

    return df.sample(frac=1.0, random_state=123).reset_index(drop=True)


def _create_labeled_df(
    gt_df: pd.DataFrame,
    pct_true_edges: float = 0.15,
    true_false_ratio: float = 2.0,
    seed: int = 123,
    *,
    tf_name_to_idx,
    tg_id_to_idx,
):
    gt_df = gt_df[
        gt_df["Source"].isin(tf_name_to_idx.keys()) &
        gt_df["Target"].isin(tg_id_to_idx.keys())
    ].copy()
    
    true_edges, false_edges = create_true_false_edges_from_full_universe(
        edge_df=gt_df,
        tf_col="Source",
        item_col="Target",
        pct_true_edges=pct_true_edges,
        true_false_ratio=true_false_ratio,
        seed=seed,
    )

    return create_labeled_tf_tg_dataset(
        true_interactions=true_edges,
        false_interactions=false_edges,
        tf_name_to_idx=tf_name_to_idx,
        drop_missing=False,
    )


# -----------------------------------
# ARGUMENT HANDLING
# -----------------------------------
sample_names = ["E7.5_rep1"]
tissues = ["mESC"]
species = "mm10"
max_peaks_per_tg = 25
max_cells_per_pair = 64
pct_true_edges = 1
true_false_ratio = 2.0
peak_flank_size = 10128
num_cpu = 4
force_reload = False
build_resample_matrices_only = False

# Only use peaks from standard chromosomes (chr1-chr19 for mm10, chr1-chr22 for hg38) to avoid issues with 
# non-standard chromosomes and contigs
if species == "mm10":
    valid_chroms = {f"chr{i}" for i in range(1, 20)}
    
    train_chroms = [str(i) for i in range(1, 16)]
    val_chroms = [ str(i) for i in range(16, 18)]
    test_chroms = [str(i) for i in range(18, 20)]
    
    gene_ref_file = DATA_DIR / "genome_data" / "genome_annotation" / "mm10" / "Mus_musculus.GRCm39.115.gtf.gz"
    
    
elif species == "hg38":
    valid_chroms = {f"chr{i}" for i in range(1, 23)}
    
    train_chroms = [str(i) for i in range(1, 18)]
    val_chroms = [str(i) for i in range(18, 20)]
    test_chroms = [str(i) for i in range(20, 23)]
    
    gene_ref_file = DATA_DIR / "genome_data" / "genome_annotation" / "hg38" / "Homo_sapiens.GRCh38.113.gtf.gz"

# Global cell type index map
cell_type_to_idx = {}

# Global TG index map
tg_id_to_idx = {}

labeled_dfs = []

cell_pools = {}

logging.info(f" === Species: {species} ===\n")

# Start of the per-sample for loop in the real dataset caching script
sample_name = sample_names[0]
tissue = tissues[0]


INFO:root: === Species: mm10 ===



### Set directories

In [128]:
logging.info(f" Building for Sample: {sample_name}, Tissue: {tissue}\n")
    
genome_fasta_path = DATA_DIR / "genome_data" / "reference_genome" / species / f"{species}.fa"
chrom_sizes_path = DATA_DIR / "genome_data" / "reference_genome" / species / f"{species}.chrom.sizes"

assert gene_ref_file.exists(), f"Gene reference file not found: {gene_ref_file}"
assert genome_fasta_path.exists(), f"Genome FASTA file not found: {genome_fasta_path}"
assert chrom_sizes_path.exists(), f"Chromosome sizes file not found: {chrom_sizes_path}"

input_data_dir = DATA_DIR / "processed" / tissue / sample_name
assert input_data_dir.exists(), f"Input data directory does not exist: {input_data_dir}"

training_cache_dir = PROJECT_DIR / "cached_data_cell_type_specific" / species / f"{tissue}_tf_tg_cache"
tf_dna_input_cache_dir = PROJECT_DIR / "cached_data" / species / "tf_dna_cache"
tf_tg_input_cache_dir = training_cache_dir / sample_name

tf_tg_input_cache_dir.mkdir(parents=True, exist_ok=True)

tf_name_to_idx_cache_path = tf_dna_input_cache_dir / "tf_name_to_idx.csv"
tf_embedding_cache_path = tf_dna_input_cache_dir / "tf_embeddings.pt"
tf_mask_cache_path = tf_dna_input_cache_dir / "tf_masks.pt"
        
atac_peak_onehot_cache_path = tf_tg_input_cache_dir / "atac_peak_tensor.pt"
train_file = tf_tg_input_cache_dir / "tftg_inputs_train.pt"
val_file = tf_tg_input_cache_dir / "tftg_inputs_val.pt"
test_file = tf_tg_input_cache_dir / "tftg_inputs_test.pt"

metadata_file = tf_tg_input_cache_dir / "metadata.json"
manifest_file = tf_tg_input_cache_dir / "manifest.json"

tss_file = DATA_DIR / "genome_data" / "genome_annotation" / species / f"{species}_gene_tss.bed"

required_cache_files = [
    tf_name_to_idx_cache_path,
    tf_embedding_cache_path,
    tf_mask_cache_path,
    atac_peak_onehot_cache_path,
    train_file,
    val_file,
    test_file,
    metadata_file,
    manifest_file,
]

cell_type_specific_gt_dir = DATA_DIR / "ground_truth_files" / "cell_type_specific"


INFO:root: Building for Sample: E7.5_rep1, Tissue: mESC



### Load the muon data

In [152]:
# -----------------------------------
# DATA LOADING
# -----------------------------------

# Load the processed Muon object
mdata = mu.read(input_data_dir / "multiome_processed.h5mu")
logging.info(f"Loaded MuData object:")
logging.info(f"  {mdata.n_obs:,} cells")
logging.info(f"  {mdata.mod['rna'].n_vars:,} genes.")
logging.info(f"  {mdata.mod['atac'].n_vars:,} peaks.")

mdata

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
INFO:root:Loaded MuData object:
INFO:root:  6,832 cells
INFO:root:  2,608 genes.
INFO:root:  1

MuData object with n_obs × n_vars = 6832 × 18853
  obs:	'leiden'
  var:	'gene_ids', 'feature_types', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
  uns:	'leiden', 'leiden_colors', 'mofa', 'neighbors', 'umap'
  obsm:	'X_mofa', 'X_umap'
  varm:	'LFs'
  obsp:	'connectivities', 'distances'
  2 modalities
    rna:	6832 x 2608
      obs:	'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'outlier', 'mt_outlier', 'doublet_score', 'predicted_doublet', 'leiden_res0_25', 'leiden_res0_5', 'leiden_res1', 'celltype', 'leiden_joint'
      var:	'gene_ids', 'feature_types', 'mt', 'ribo', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      uns:	'celltype_colors', 'hvg', 'leiden_res0_25', 'leiden_res0_25_colors', 'leiden_res0_5', 'leiden_res0_5_colors', 'leiden_res1', 'leiden_res1_colors', 'log1p', 'neighbors', 'pca', 'scrublet', 'tsne', 'umap'
      obsm:	'X_pca', 'X_tsne', 'X_umap'
      varm:	'PCs'
      layers:	'counts'
      obsp:	'connectivities', 'distances'
    atac:	6832 x 16245
      obs:	'n_genes_by_counts', 'total_counts', 'outlier', 'NS', 'n_counts', 'leiden', 'leiden_joint'
      var:	'gene_ids', 'feature_types', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std', 'nearest_gene', 'TSS_dist', 'TSS_dist_score'
      uns:	'files', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'lsi', 'neighbors', 'pca', 'peak_gene_distance_params', 'umap'
      obsm:	'X_lsi', 'X_pca', 'X_umap'
      varm:	'LSI', 'PCs'
      layers:	'counts'
      obsp:	'connectivities', 'distances'

### Save the RNA and ATAC matrices to cache for the sample

In [153]:
from scipy import sparse

# tf_tg_input_cache_dir already includes the sample in your notebook.
cache_dir = tf_tg_input_cache_dir
cache_dir.mkdir(parents=True, exist_ok=True)


def cache_matrix(matrix, name, cache_dir, force=False):
    if sparse.issparse(matrix):
        matrix = matrix.tocsr().astype(np.float32)
        path = cache_dir / f"{name}.npz"

        if force or not path.exists():
            sparse.save_npz(path, matrix)
    else:
        path = cache_dir / f"{name}.pt"

        if force or not path.exists():
            torch.save(
                torch.as_tensor(matrix, dtype=torch.float32),
                path,
            )

    return path


atac_mat = mdata.mod["atac"].X
rna_mat = mdata.mod["rna"].X

atac_mat_cache_path = cache_matrix(
    atac_mat, "atac_mat", cache_dir, force=force_reload
)
rna_mat_cache_path = cache_matrix(
    rna_mat, "rna_mat", cache_dir, force=force_reload
)

### Filter the mdata to the cell type

In [154]:
cell_type = mdata.mod["rna"].obs["celltype"].unique()[1]
print(cell_type)

mdata_ct = mdata[mdata.mod["rna"].obs["celltype"] == cell_type]

rna_adata = mdata_ct.mod["rna"]
atac_adata = mdata_ct.mod["atac"]

assert rna_adata.obs_names.is_unique
assert atac_adata.obs_names.is_unique

# Preserve RNA order and retain paired cells.
shared_cells = rna_adata.obs_names[rna_adata.obs_names.isin(atac_adata.obs_names)]
rna = rna_adata[shared_cells].copy()
atac = atac_adata[shared_cells].copy()

assert rna.obs_names.equals(atac.obs_names)

cell_pools[(sample_name, cell_type)] = np.flatnonzero(
    rna.obs["celltype"].eq(cell_type).fillna(False).to_numpy()
)

# Uppercase gene names in mdata
rna_adata.var_names = rna_adata.var_names.str.upper()
    
# Uppercase gene names in the peak-to-gene distance variable
atac_adata.var["nearest_gene"] = atac_adata.var["nearest_gene"].str.upper()

ExE_ectoderm


/tmp/ipykernel_1032160/576596572.py:27: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  atac_adata.var["nearest_gene"] = atac_adata.var["nearest_gene"].str.upper()


### Load the ground truth file for the cell type

In [155]:
logging.info(f"Building TF-TG training data for cell type: {cell_type}")

celltype_df_file = cell_type_specific_gt_dir / species / tissue / f"{cell_type}_ground_truth.parquet"
assert celltype_df_file.exists(), \
    f"Ground truth file for cell type '{cell_type}' not found: {celltype_df_file}"

cell_type_gt_df = pd.read_parquet(celltype_df_file)
cell_type_gt_df.head()
    

INFO:root:Building TF-TG training data for cell type: ExE_ectoderm


,Source,Target,cell_type,source,source_term,proxy_tier
0,GATA3,GM28310,ExE_ectoderm,chipatlas,Induced_trophoblast_stem_cells,2
1,TFAP2C,GM21241,ExE_ectoderm,chipatlas,Induced_trophoblast_stem_cells,2
2,GATA3,GM57922,ExE_ectoderm,chipatlas,Induced_trophoblast_stem_cells,2
3,SOX2,GM58035,ExE_ectoderm,chipatlas,Induced_trophoblast_stem_cells,2
4,TFAP2C,GM58035,ExE_ectoderm,chipatlas,Induced_trophoblast_stem_cells,2


### Data Filtering

In [156]:
# -----------------------------------
# DATA FILTERING
# -----------------------------------
gt_tfs_in_rna = set(cell_type_gt_df["Source"]).intersection(rna_adata.var_names)
gt_tgs_in_rna = set(cell_type_gt_df["Target"]).intersection(rna_adata.var_names)
logging.info(f"Ground truth TFs in RNA pseudobulk: {len(gt_tfs_in_rna)} (Example: {list(gt_tfs_in_rna)[:5]})")
logging.info(f"Ground truth TGs in RNA pseudobulk: {len(gt_tgs_in_rna)} (Example: {list(gt_tgs_in_rna)[:5]})")

n_before_rna_filter = len(cell_type_gt_df)

# Subset the ground truth to only TFs and TGs present in the RNA data
cell_type_gt_df = cell_type_gt_df[
    cell_type_gt_df["Source"].isin(gt_tfs_in_rna) &
    cell_type_gt_df["Target"].isin(gt_tgs_in_rna)
].copy()

logging.info(
    f"Ground truth edges after RNA TF/TG filtering: "
    f"{len(cell_type_gt_df):,} / {n_before_rna_filter:,}"
)

# TF name to index mapping
tf_name_to_idx = pd.read_csv(tf_name_to_idx_cache_path)
tf_name_to_idx["tf_name"] = tf_name_to_idx["tf_name"].str.upper()
tf_name_to_idx = tf_name_to_idx.set_index("tf_name")["tf_idx"].to_dict()

# ATAC peak to index mapping (only keep peaks on valid chromosomes)
dataset_peaks = atac_adata.var_names.to_list()
dataset_peaks = [peak for peak in dataset_peaks if peak.split(":", 1)[0] in valid_chroms]

# Filter ground truth to only TFs with embeddings
# (i.e. were present in the TF-DNA model training data)
gt_tfs_in_embeddings = set(tf_name_to_idx.keys()).intersection(gt_tfs_in_rna)
logging.info(f"Ground truth TFs with embeddings: {len(gt_tfs_in_embeddings)} (Example: {list(gt_tfs_in_embeddings)[:5]})")

n_before_tf_embedding_filter = len(cell_type_gt_df)

cell_type_gt_df = cell_type_gt_df[
    cell_type_gt_df["Source"].isin(gt_tfs_in_embeddings)
].copy()

logging.info(
    f"Ground truth edges after filtering to TFs with embeddings: "
    f"{len(cell_type_gt_df):,} / {n_before_tf_embedding_filter:,}"
)

# Add any unseen TGs to the global TG index map
for tg in cell_type_gt_df["Target"].dropna().unique():
    if tg not in tg_id_to_idx:
        tg_id_to_idx[tg] = len(tg_id_to_idx)

tf_rna_col_map = {tf: idx for idx, tf in enumerate(rna_adata.var_names)}
tg_rna_col_map = {tg: idx for idx, tg in enumerate(rna_adata.var_names)}
cell_col_map = {cell: idx for idx, cell in enumerate(rna_adata.obs_names)}
atac_peak_map = {peak: idx for idx, peak in enumerate(dataset_peaks)}

idx_to_cell = {idx: cell for cell, idx in cell_col_map.items()}

INFO:root:Ground truth TFs in RNA pseudobulk: 16 (Example: ['KLF5', 'PRDM1', 'IRF2', 'ESRRB', 'EOMES'])
INFO:root:Ground truth TGs in RNA pseudobulk: 2311 (Example: ['CLTC', 'FOXO1', 'PTPRG', 'MAML2', 'NAALADL2'])
INFO:root:Ground truth edges after RNA TF/TG filtering: 14,315 / 444,448
INFO:root:Ground truth TFs with embeddings: 16 (Example: ['KLF5', 'PRDM1', 'ESRRB', 'IRF2', 'EOMES'])
INFO:root:Ground truth edges after filtering to TFs with embeddings: 14,315 / 14,315


### Create labeled edge DataFrame

In [157]:
tf_tg_labeled_df = _create_labeled_df(
    cell_type_gt_df,
    pct_true_edges,
    true_false_ratio,
    seed=123,
    tf_name_to_idx=tf_name_to_idx,
    tg_id_to_idx=tg_rna_col_map
)

tf_tg_labeled_df["sample_id"] = sample_name
tf_tg_labeled_df["cell_type"] = cell_type
tf_tg_labeled_df["tf_rna_col"] = tf_tg_labeled_df["tf_name"].map(tf_rna_col_map)
tf_tg_labeled_df["tg_rna_col"] = tf_tg_labeled_df["tg_id"].map(tg_rna_col_map)

col_order = ["sample_id", "cell_type", "tf_name", "tg_id", "tf_embedding_idx", "tf_rna_col", "tg_rna_col", "label"]
tf_tg_labeled_df = tf_tg_labeled_df[col_order]

# labeled_dfs.append(tf_tg_labeled_df)

tf_tg_labeled_df

,sample_id,cell_type,tf_name,tg_id,tf_embedding_idx,tf_rna_col,tg_rna_col,label
0,E7.5_rep1,ExE_ectoderm,IRF2,WFDC2,313,1160,368,False
1,E7.5_rep1,ExE_ectoderm,MEIS1,TACC1,257,1555,1140,True
2,E7.5_rep1,ExE_ectoderm,POU3F1,TTLL12,303,612,2100,False
3,E7.5_rep1,ExE_ectoderm,TBX20,CNTN5,416,1272,1265,False
4,E7.5_rep1,ExE_ectoderm,TFAP2C,STK17B,17,379,45,True
...,...,...,...,...,...,...,...,...
34923,E7.5_rep1,ExE_ectoderm,MEIS1,PACSIN1,257,1555,2228,True
34924,E7.5_rep1,ExE_ectoderm,ESRRB,GRIN1,360,1790,213,False
34925,E7.5_rep1,ExE_ectoderm,GATA2,MED13L,255,920,780,False
34926,E7.5_rep1,ExE_ectoderm,PRDM1,CCNE2,130,1445,522,False


### Train / Val / Test splits by chromosome

In [158]:
# Split genes into train/val/test based on chromosome using the GTF reference file
train_genes, val_genes, test_genes = split_genes_by_chromosome(
    gene_ref_file,
    train_chroms=train_chroms,
    val_chroms=val_chroms,
    test_chroms=test_chroms
    )

INFO:root:Splitting genes into train/val/test based on chromosome:
INFO:root:Extracted GTF attributes: ['gene_id', 'gene_version', 'gene_name', 'gene_source', 'gene_biotype', 'transcript_id', 'transcript_version', 'transcript_name', 'transcript_source', 'transcript_biotype', 'tag', 'transcript_support_level', 'exon_number', 'exon_id', 'exon_version', 'protein_id', 'protein_version', 'ccds_id']
INFO:root:  - Train set: 62,426 genes (chroms 1-15)
INFO:root:  - Validation set: 5,700 genes (chroms 16-17)
INFO:root:  - Test set: 4,099 genes (chroms 18-19)


In [159]:
# Subset the ground truth to create train/val/test splits based on the target gene chromosome splits
# (Only keeps TFs and TGs present in the ground truth and RNA pseudobulk, and only keeps TFs with embeddings)
gt_train_df, gt_val_df, gt_test_df = create_train_val_test_splits(
    tf_tg_labeled_df, train_genes, val_genes, test_genes
)

logging.info(f"After subsetting to TFs with embeddings and TGs in RNA pseudobulk:")
logging.info(f"  - Train interactions: {len(gt_train_df)} (TFs: {gt_train_df['tf_name'].nunique()}, TGs: {gt_train_df['tg_id'].nunique()})")
logging.info(f"  - Val interactions: {len(gt_val_df)} (TFs: {gt_val_df['tf_name'].nunique()}, TGs: {gt_val_df['tg_id'].nunique()})")
logging.info(f"  - Test interactions: {len(gt_test_df)} (TFs: {gt_test_df['tf_name'].nunique()}, TGs: {gt_test_df['tg_id'].nunique()})")


INFO:root:After subsetting to TFs with embeddings and TGs in RNA pseudobulk:
INFO:root:  - Train interactions: 28752 (TFs: 16, TGs: 1797)
INFO:root:  - Val interactions: 2416 (TFs: 16, TGs: 151)
INFO:root:  - Test interactions: 2400 (TFs: 16, TGs: 150)


### ATAC peak centered one-hot encodings

In [160]:
# Create or load cached one-hot encodings for ATAC peaks
# One-hot encodings use ACGT order and uses 'flank_size' bp upstream and downstream of the peak center.    
if os.path.exists(atac_peak_onehot_cache_path):
    atac_peak_tensor = torch.load(atac_peak_onehot_cache_path, weights_only=True)
    
    expected_n_peaks = len(dataset_peaks)

    if atac_peak_tensor.shape[0] != expected_n_peaks:
        raise ValueError(
            f"ATAC one-hot tensor has {atac_peak_tensor.shape[0]:,} peaks, "
            f"but current dataset_peaks has {expected_n_peaks:,}. "
            "Delete the cached ATAC peak tensor or rerun with --force_reload."
        )
    
else:
    logging.info("Creating centered peak one-hot encodings for ATAC peaks...")
    atac_peak_array = utils.create_centered_peak_onehot_array(
        peak_ids=dataset_peaks,
        genome_fasta=genome_fasta_path,
        chrom_sizes=utils.load_chrom_sizes(chrom_sizes_path),
        peak_id_to_idx=atac_peak_map,
        flank_size=peak_flank_size,
        dtype=np.uint8,
        pad_out_of_bounds=True,
        num_workers=num_cpu,
        show_progress=True,
        chunk_size=10000,
    )
    atac_peak_tensor = torch.as_tensor(atac_peak_array, dtype=torch.uint8)
    atac_peak_tensor = atac_peak_tensor.float()
    torch.save(atac_peak_tensor, atac_peak_onehot_cache_path)
    
if atac_peak_tensor.dtype == torch.uint8:
    atac_peak_tensor = atac_peak_tensor.float()


### Add the peak ID and distances to the labeled dataframe

In [161]:
# -----------------------------------
# PREPARE LOOKUP TABLES
# -----------------------------------
# TG to peak information map (peak IDs, indices, and distances to TSS)
tg_to_peak_info = {}
for tg in atac_adata.var["nearest_gene"].dropna().unique():
    tg_norm = tg.upper()
    
    peak_df = atac_adata.var.loc[
        atac_adata.var["nearest_gene"].eq(tg),
        ["TSS_dist"],
    ].copy()
    
    peak_df["abs_distance"] = peak_df["TSS_dist"].abs()

    peak_df = (
        peak_df
        .sort_values("abs_distance", kind="stable")
        .head(max_peaks_per_tg)
    )

    peak_ids = peak_df.index.to_list()    
    
    peak_distances = peak_df.loc[peak_ids, "TSS_dist"].to_list()
    
    peak_col_idxs = [atac_peak_map.get(peak, None) for peak in peak_ids]

    tg_to_peak_info[tg_norm] = {
        "peak_ids": peak_ids,
        "peak_distances": peak_distances,
        "peak_col_idxs": peak_col_idxs,
    }

tf_tg_labeled_df["peak_ids"] = tf_tg_labeled_df["tg_id"].map(lambda tg: tg_to_peak_info.get(tg, {}).get("peak_ids", []))
tf_tg_labeled_df["peak_atac_cols"] = tf_tg_labeled_df["tg_id"].map(lambda tg: tg_to_peak_info.get(tg, {}).get("peak_col_idxs", []))
tf_tg_labeled_df["peak_distances"] = tf_tg_labeled_df["tg_id"].map(lambda tg: tg_to_peak_info.get(tg, {}).get("peak_distances", []))

In [162]:
tf_tg_labeled_df

,sample_id,cell_type,tf_name,tg_id,tf_embedding_idx,tf_rna_col,tg_rna_col,label,peak_ids,peak_atac_cols,peak_distances
0,E7.5_rep1,ExE_ectoderm,IRF2,WFDC2,313,1160,368,False,[],[],[]
1,E7.5_rep1,ExE_ectoderm,MEIS1,TACC1,257,1555,1140,True,[chr8:25719748-25720624],[14263],[10255]
2,E7.5_rep1,ExE_ectoderm,POU3F1,TTLL12,303,612,2100,False,[chr15:83481563-83482381],[5423],[2204]
3,E7.5_rep1,ExE_ectoderm,TBX20,CNTN5,416,1272,1265,False,[],[],[]
4,E7.5_rep1,ExE_ectoderm,TFAP2C,STK17B,17,379,45,True,[],[],[]
...,...,...,...,...,...,...,...,...,...,...,...
34923,E7.5_rep1,ExE_ectoderm,MEIS1,PACSIN1,257,1555,2228,True,[chr17:27907243-27908093],[6418],[3029]
34924,E7.5_rep1,ExE_ectoderm,ESRRB,GRIN1,360,1790,213,False,"[chr2:25193095-25194009, chr2:25185665-2518642...","[8161, 8160, 8162, 8163, 8164]","[0, 262, 1690, 3985, 5059]"
34925,E7.5_rep1,ExE_ectoderm,GATA2,MED13L,255,920,780,False,"[chr5:118691081-118691959, chr5:118866156-1188...","[11818, 11821]","[6785, 18297]"
34926,E7.5_rep1,ExE_ectoderm,PRDM1,CCNE2,130,1445,522,False,[],[],[]


### Check the TGs with peaks and how many peaks the TG with the most peaks has

In [163]:
if tf_tg_labeled_df.empty:
    raise ValueError(
        "No labeled TF-TG pairs were created across train/val/test. "
        "Check RNA filtering, TF embedding filtering, chromosome splits, and ground truth overlap."
    )

max_peaks_real = max(
    len(tg_to_peak_info.get(tg_name, {}).get("peak_indices", []))
    for tg_name in tf_tg_labeled_df["tg_id"]
)

# Check that at least some TGs have peaks within 100kb, otherwise the model will have no signal to learn from
n_tgs_with_peaks = sum(
    len(tg_to_peak_info.get(tg, {}).get("peak_indices", [])) > 0
    for tg in tf_tg_labeled_df["tg_id"].unique()
)

logging.info(f"TGs with at least one peak within 100kb: {n_tgs_with_peaks:,} / {tf_tg_labeled_df['tg_id'].nunique():,}")
logging.info(f"Max peaks per TG after filtering/capping: {max_peaks_real:,}")

if max_peaks_real == 0:
    raise ValueError(
        "No labeled TGs have peaks within 100kb. Check target_id_norm/tg_id matching, "
        "peak IDs, chromosome filtering, and TSS distance file."
    )

INFO:root:TGs with at least one peak within 100kb: 0 / 2,183
INFO:root:Max peaks per TG after filtering/capping: 0


ValueError: No labeled TGs have peaks within 100kb. Check target_id_norm/tg_id matching, peak IDs, chromosome filtering, and TSS distance file.

### Build the train/val/test datasets for the cell type

In [ ]:
cell_names = list(rna_adata.obs_names)

cell_indices = np.asarray([cell_col_map[c] for c in cell_names], dtype=np.int64)

tg_peak_info = {}
tf_indices = []
tg_indices = []

labels = []

rng = np.random.default_rng(42)

# Sample a subset of cells for each TF-TG pair
for row in gt_train_df.itertuples(index=False):
    label = row.label
    
    tf_idx = row.tf_idx
    tg_idx = row.tg_idx
    ct_idx = row.ct_idx
    
    # Create a bag of peak information for each TG 
    # (shared per TG, so only created once per TG for each cell type)
    if tg_idx not in tg_peak_info:
        peak_info = tg_to_peak_info.get(tg_idx)
        
        peak_indices_real = list(peak_info["peak_indices"])
        peak_dst_real = list(peak_info["peak_distances"])
        n_peaks = len(peak_indices_real)
        
        peak_indices = np.asarray(peak_indices_real, dtype=np.int64)
        peak_distances = np.asarray(peak_dst_real, dtype=np.float32)
        peak_mask = np.ones(n_peaks, dtype=bool)
        
        # Pad to max peaks per TG
        if n_peaks < max_peaks_real:
            pad_length = max_peaks_real - n_peaks
            peak_indices = np.pad(peak_indices, (0, pad_length), constant_values=-1)
            peak_distances = np.pad(peak_distances, (0, pad_length), constant_values=0.0)
            peak_mask = np.pad(peak_mask, (0, pad_length), constant_values=False)
        
        # Shared by every edge on this TG from here on, and np.stack copies them into
        # the output, so freeze them rather than trust that nobody writes through.
        for arr in (peak_indices, peak_distances, peak_mask):
            arr.flags.writeable = False
        
        # Create a bag of peak information for each TG
        bag = (peak_indices, peak_distances, peak_mask, len(peak_indices),
                np.asarray(peak_indices, dtype=np.int64))
        tg_peak_info[tg_idx] = bag
    else:
        bag = tg_peak_info[tg_idx]
    
    
    tf_indices.append(tf_idx)
    tg_indices.append(tg_idx)

    labels.append(label)

# Build the input data structure for the model
n_tg_total = len(tg_id_to_idx)

tg_peak_indices_arr = np.zeros((n_tg_total, max_peaks_real), dtype=np.int64)
tg_peak_distance_arr = np.zeros((n_tg_total, max_peaks_real), dtype=np.float32)
tg_peak_mask_arr = np.zeros((n_tg_total, max_peaks_real), dtype=bool)

for tg_idx, bag in tg_peak_info.items():
    tg_row = tg_id_to_idx.get(tg_idx)
    peak_idx_row, peak_dst_row, peak_mask_row, _, _ = bag
    tg_peak_indices_arr[tg_row] = peak_idx_row
    tg_peak_distance_arr[tg_row] = peak_dst_row
    tg_peak_mask_arr[tg_row] = peak_mask_row


data_dict = {
    # Cell type specific labels for each edge in the dataset 
    # (1 for true interaction, 0 for false interaction)
    "label": torch.tensor(labels, dtype=torch.float32),
    
    # Indices for TFs, TGs, and cells for each edge in the dataset
    "tf_idx": torch.tensor(tf_indices, dtype=torch.long),
    "tg_idx": torch.tensor(tg_indices, dtype=torch.long),
    "cell_indices": torch.tensor(cell_indices, dtype=torch.long),

    # Per-TG peak information for the cell type (shared across all edges for a given TG)
    "tg_peak_indices": torch.tensor(tg_peak_indices_arr, dtype=torch.long),
    "tg_peak_distance": torch.tensor(tg_peak_distance_arr, dtype=torch.float32),
    "tg_peak_mask": torch.tensor(tg_peak_mask_arr, dtype=torch.bool),
}


        

For each input, I need:

```
label:            [T]
tf_idx:           [E]
tg_idx:           [E]
cell_indices:     [T, E, C]   int64 column indices into atac_mat / rna_mat
tg_peak_indices:  [T, G, P]   G = len(tg_id_to_idx); gather rows via tg_idx
tg_peak_distance: [T, G, P]
tg_peak_mask:     [T, G, P]
cell_type:        [T]      (New)
```

In [146]:
from scipy import sparse
from torch.utils.data import Dataset


class TFTGEdgeBagDataset(Dataset):
    """One dataset instance per sample and cell type."""

    def __init__(
        self,
        inputs,  # Your labeled edge DataFrame
        *,
        tf_embeddings_tensor,
        tf_mask_tensor,
        atac_peak_tensor,
        atac_mat,
        rna_mat,
        cell_pool,
        max_peaks_per_tg=25,
        resample_max_cells_per_pair=64,
        resample_cells=True,
        seed=42,
    ):
        self.inputs = inputs.reset_index(drop=True).copy()

        self.tf_embeddings_tensor = tf_embeddings_tensor
        self.tf_mask_tensor = tf_mask_tensor
        self.atac_peak_tensor = atac_peak_tensor

        # Keep sparse matrices sparse
        self.atac_mat = atac_mat.tocsr() if sparse.issparse(atac_mat) else atac_mat
        self.rna_mat = rna_mat.tocsr() if sparse.issparse(rna_mat) else rna_mat

        self.cell_pool = np.asarray(cell_pool, dtype=np.int64)
        self.C = int(resample_max_cells_per_pair)
        self.P = int(max_peaks_per_tg)
        self.resample_cells = resample_cells
        self.seed = int(seed)
        self._rng = None

        if self.C <= 0 or self.P <= 0:
            raise ValueError("Cell and peak limits must be positive.")

        if self.rna_mat.shape[0] != self.atac_mat.shape[0]:
            raise ValueError("RNA and ATAC must have aligned cell rows.")

        if self.cell_pool.ndim != 1 or len(self.cell_pool) == 0:
            raise ValueError("cell_pool must be a nonempty 1D array.")

        if len(np.unique(self.cell_pool)) != len(self.cell_pool):
            raise ValueError("cell_pool contains duplicate positions.")

        if (
            self.cell_pool.min() < 0
            or self.cell_pool.max() >= self.rna_mat.shape[0]
        ):
            raise ValueError("cell_pool contains out-of-range cell positions.")

        if self.atac_peak_tensor.shape[0] != self.atac_mat.shape[1]:
            raise ValueError(
                "Peak sequences and ATAC columns must use the same peak order."
            )

    def __len__(self):
        return len(self.inputs)

    @staticmethod
    def _gather(matrix, rows, cols):
        """Gather a cells × features slice as a float tensor."""
        if torch.is_tensor(matrix):
            rows = torch.as_tensor(rows, dtype=torch.long, device=matrix.device)
            cols = torch.as_tensor(cols, dtype=torch.long, device=matrix.device)
            return matrix.index_select(0, rows).index_select(1, cols).float()

        block = matrix[rows][:, cols]
        if sparse.issparse(block):
            block = block.toarray()

        return torch.as_tensor(np.asarray(block), dtype=torch.float32)

    def __getitem__(self, idx):
        row = self.inputs.iloc[idx]
        tf_idx = int(row.tf_embedding_idx)

        # These lists must already be sorted/capped together during preparation.
        peak_cols = np.asarray(row.peak_atac_cols, dtype=np.int64)
        distances = np.asarray(row.peak_distances, dtype=np.float32)

        n_peaks = len(peak_cols)
        if len(distances) != n_peaks or not 0 < n_peaks <= self.P:
            raise ValueError(f"Invalid peak bag for edge {idx}.")

        if np.any(peak_cols < 0) or np.any(peak_cols >= self.atac_mat.shape[1]):
            raise ValueError(f"Invalid ATAC column for edge {idx}.")

        # Training: fresh draws. Validation/test: fixed draw for each edge.
        if self.resample_cells:
            if self._rng is None:
                self._rng = np.random.default_rng(
                    np.random.SeedSequence([self.seed, torch.initial_seed()])
                )
            rng = self._rng
        else:
            rng = np.random.default_rng(
                np.random.SeedSequence([self.seed, int(idx)])
            )

        n_cells = min(self.C, len(self.cell_pool))
        cell_rows = rng.choice(
            self.cell_pool,
            size=n_cells,
            replace=False,
        )

        # Fixed-size cell bag.
        cell_indices = torch.full((self.C,), -1, dtype=torch.long)
        cell_indices[:n_cells] = torch.from_numpy(cell_rows)

        cell_mask = torch.zeros(self.C, dtype=torch.bool)
        cell_mask[:n_cells] = True

        # Fixed-size peak bag.
        peak_indices = torch.full((self.P,), -1, dtype=torch.long)
        peak_indices[:n_peaks] = torch.from_numpy(peak_cols)

        peak_mask = torch.zeros(self.P, dtype=torch.bool)
        peak_mask[:n_peaks] = True

        peak_distance = torch.zeros(self.P, dtype=torch.float32)
        peak_distance[:n_peaks] = torch.from_numpy(distances)

        # Only gather real sequences; never index padding with -1.
        peak_sequences = torch.zeros(
            (self.P, *self.atac_peak_tensor.shape[1:]),
            dtype=torch.float32,
        )
        peak_sequences[:n_peaks] = self.atac_peak_tensor[
            torch.from_numpy(peak_cols)
        ].float()

        # Same selected cells across all features.
        peak_accessibility = torch.zeros(self.C, self.P)
        peak_accessibility[:n_cells, :n_peaks] = self._gather(
            self.atac_mat, cell_rows, peak_cols
        )

        tf_expression = torch.zeros(self.C)
        tf_expression[:n_cells] = self._gather(
            self.rna_mat, cell_rows, [int(row.tf_rna_col)]
        ).flatten()

        tg_expression = torch.zeros(self.C)
        tg_expression[:n_cells] = self._gather(
            self.rna_mat, cell_rows, [int(row.tg_rna_col)]
        ).flatten()

        return {
            "sample_id": row.sample_id,
            "cell_type": row.cell_type,
            "tf_name": row.tf_name,
            "tg_id": row.tg_id,
            "label": torch.tensor(float(row.label)),
            "tf_idx": torch.tensor(tf_idx, dtype=torch.long),
            "cell_indices": cell_indices,
            "cell_mask": cell_mask,
            "peak_indices": peak_indices,
            "peak_sequences": peak_sequences,
            "peak_distance": peak_distance,
            "peak_mask": peak_mask,
            "peak_accessibility": peak_accessibility,
            "tf_expression": tf_expression,
            "tg_expression": tg_expression,
            "tf_embedding": self.tf_embeddings_tensor[tf_idx].float(),
            "tf_mask": self.tf_mask_tensor[tf_idx].bool(),
        }

In [148]:
cell_pools

{}

In [150]:
tf_embeddings_tensor = torch.load(tf_embedding_cache_path, weights_only=True)
tf_mask_tensor = torch.load(tf_mask_cache_path, weights_only=True)

train_dataset = TFTGEdgeBagDataset(
    inputs=gt_train_df,
    tf_embeddings_tensor=tf_embeddings_tensor,
    tf_mask_tensor=tf_mask_tensor,
    atac_peak_tensor=atac_peak_tensor,
    atac_mat=atac_mat,
    rna_mat=rna_mat,
    cell_pool=cell_pools[(sample_name, cell_type)],
    max_peaks_per_tg=25,
    resample_max_cells_per_pair=64,
    resample_cells=True,
)

ValueError: Peak sequences and ATAC columns must use the same peak order.